# Avant le TP2 : créer, alimenter et lire un topic en Python

Le TP2 utilise trois objets de `confluent-kafka` : l'`AdminClient`, le producteur et le consommateur. Cette démonstration montre comment chacun répond, sur un topic jetable, `demo-tp2` :

1. une opération d'administration renvoie un **futur**, et ses erreurs arrivent par une exception ;
2. le producteur met les messages en file : le **rapport de livraison** dit s'ils sont écrits ;
3. le consommateur peut lire une partition précise à partir d'un offset.

Sorties de référence : exécution du 07/11/2026 22:41 UTC sur le lab.

In [1]:
import os
import time

from confluent_kafka import Consumer, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic

BS = os.environ["BS"]
admin = AdminClient({"bootstrap.servers": BS})

## 1. Administration : des futurs

`create_topics()` ne bloque pas : il renvoie un dictionnaire `{nom: futur}`. `result()` attend la réponse du cluster.

In [2]:
futurs = admin.create_topics([NewTopic("demo-tp2", num_partitions=2, replication_factor=3)])
print(futurs)

futurs["demo-tp2"].result()
print("demo-tp2 cree")

{'demo-tp2': <Future at 0xffff8eca9450 state=running>}
demo-tp2 cree


Une erreur du cluster est levée par `result()` sous forme de `KafkaException`. Son premier argument porte le code d'erreur, le même que celui des outils Java.

In [3]:
try:
    admin.create_topics([NewTopic("demo-tp2", num_partitions=2, replication_factor=3)])["demo-tp2"].result()
except KafkaException as e:
    print("Code :", e.args[0].name())
    print("Message :", e.args[0].str())

Code : TOPIC_ALREADY_EXISTS
Message : Topic 'demo-tp2' already exists.


## 2. Producteur : la file et le rapport de livraison

`produce()` range le message dans une file et rend la main tout de suite. `flush()` envoie la file et attend les réponses ; le rappel `on_delivery` est alors appelé une fois par message, avec l'erreur (ou `None`) et le message écrit.

In [4]:
producteur = Producer({"bootstrap.servers": BS, "log_level": 4})


def rapport(err, msg):
    if err is not None:
        print("Echec :", err)
    else:
        print("cle %s -> partition %d, offset %d" % (msg.key().decode(), msg.partition(), msg.offset()))


for cle, valeur in [("rouge", "premier"), ("vert", "deuxieme"), ("rouge", "troisieme")]:
    producteur.produce("demo-tp2", key=cle, value=valeur, on_delivery=rapport)
print("Messages en file, rien n'est encore confirme")

restant = producteur.flush(10)
print("Messages encore en file apres flush() :", restant)

Messages en file, rien n'est encore confirme


cle vert -> partition 0, offset 0
cle rouge -> partition 1, offset 0
cle rouge -> partition 1, offset 1
Messages encore en file apres flush() : 0


Les deux messages de clé `rouge` sont dans la même partition, l'un à la suite de l'autre ; celui de clé `vert` est dans l'autre. Même constat en ligne de commande :

In [5]:
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic demo-tp2

demo-tp2:0:1
demo-tp2:1:2


## 3. Consommateur : lire une partition à partir d'un offset

`assign()` lit directement les partitions indiquées, à partir de l'offset donné, sans rééquilibrage de groupe. `consume(n, timeout)` renvoie une liste d'au plus `n` messages.

In [6]:
consommateur = Consumer({"bootstrap.servers": BS, "group.id": "demo-tp2",
                         "enable.auto.commit": False, "log_level": 4})
consommateur.assign([TopicPartition("demo-tp2", 0, 0), TopicPartition("demo-tp2", 1, 0)])

for m in consommateur.consume(10, timeout=10):
    print(m.partition(), m.offset(), m.key().decode(), m.value().decode())
consommateur.close()

1 0 rouge premier
1 1 rouge troisieme
0 0 vert deuxieme


Dans une partition, les messages arrivent dans l'ordre de leurs offsets. Entre deux partitions, aucun ordre n'est garanti : c'est pourquoi la clé, qui fixe la partition, fixe aussi l'ordre.

## 4. Ménage

Le topic de démonstration est supprimé : le TP2 commence sur un cluster sans topic métier. La suppression est asynchrone : on interroge le cluster jusqu'à ce que le topic ait disparu.

In [7]:
admin.delete_topics(["demo-tp2"])["demo-tp2"].result()

for _ in range(15):
    if "demo-tp2" not in admin.list_topics(timeout=10).topics:
        break
    time.sleep(2)
print(sorted(admin.list_topics(timeout=10).topics))

['__consumer_offsets', 'loadgen']


## La méthode du TP2

Ouvrez `tps/jour1/tp2_topics/enonce.ipynb`. Ce TP crée des topics et envoie des messages : il modifie le cluster.

- Quand une commande modifie le cluster, son équivalent en ligne de commande est affiché **sans être exécuté**, avec la mention « ne pas lancer en plus du TODO ». C'est votre code Python qui fait l'opération.
- Exécutez une seule fois les cellules qui envoient des messages : les vérifications comptent les messages.
- Chaque vérification compare votre résultat à celui qu'aurait donné l'outil Java : clés, partitions, offsets.